In [ ]:
import os
import socket
import sqlite3
import sys
import time

from datetime import datetime

import requests
import urllib3.util.connection


# ==================================================
# IPv4固定
# ==================================================

def allowed_gai_family():
    return socket.AF_INET


urllib3.util.connection.allowed_gai_family = allowed_gai_family


# ==================================================
# importパス設定
# ==================================================

# Jupyter・通常のPythonスクリプトの両方に対応
try:
    base_dir = os.path.dirname(
        os.path.abspath(__file__)
    )
except NameError:
    base_dir = os.getcwd()


project_root = os.path.abspath(
    os.path.join(
        base_dir,
        "..",
        "..",
    )
)


if project_root not in sys.path:
    sys.path.insert(
        0,
        project_root,
    )


from utils.config import (
    DB_PATH,
    RAKUTEN_ACCESS_KEY,
    RAKUTEN_APPLICATION_ID,
)


# ==================================================
# 初期設定
# ==================================================

start_time = time.time()

table_name = "result_table"


# ==================================================
# 楽天市場API設定
# ==================================================

SHOP_NAME = "楽天市場"
SHOP_PRODUCT_ID = 100
CATEGORY = "slot"


API_URL = (
    "https://openapi.rakuten.co.jp/"
    "ichibams/api/IchibaItem/Search/20260701"
)


# ==================================================
# 検索条件
# ==================================================

KEYWORD = "Lスマスロ北斗の拳転生の章2"


# 1ページあたりの取得件数
# 楽天APIは最大30件
HITS = 30


# 最大取得ページ数
# 楽天APIは最大100ページ
MAX_PAGE = 2


# ページ間の待機秒数
REQUEST_INTERVAL = 2


# タイムアウト秒数
REQUEST_TIMEOUT = 120


# 1ページあたりの最大再試行回数
MAX_RETRIES = 5


# 再試行までの待機秒数
RETRY_INTERVAL = 30


# ==================================================
# IPv4確認
# ==================================================

def get_current_ipv4() -> str:

    try:

        response = requests.get(
            "https://api.ipify.org",
            timeout=30,
        )

        response.raise_for_status()

        return response.text.strip()

    except requests.RequestException as e:

        return f"取得失敗: {e}"


# ==================================================
# APIセッション
# ==================================================

def open_session() -> requests.Session:

    session = requests.Session()

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/138.0.0.0 "
            "Safari/537.36"
        ),
        "accessKey": RAKUTEN_ACCESS_KEY,
    })

    return session


# ==================================================
# API取得
# ==================================================

def get_page(
    session: requests.Session,
    page: int,
) -> requests.Response | None:

    params = {
        "applicationId": RAKUTEN_APPLICATION_ID,
        "keyword": KEYWORD,
        "hits": HITS,
        "page": page,
        "format": "json",
        "formatVersion": 2,
        "elements": (
            "itemName,"
            "itemCode,"
            "itemPrice,"
            "itemUrl,"
            "mediumImageUrls,"
            "shopName,"
            "shopCode"
        ),
    }


    for retry_count in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:

            print(
                f"[REQUEST] Page {page} 取得開始 "
                f"({retry_count}/{MAX_RETRIES})"
            )

            response = session.get(
                API_URL,
                params=params,
                timeout=REQUEST_TIMEOUT,
            )


            if response.status_code == 200:
                return response


            print(
                f"[ERROR] HTTP "
                f"{response.status_code}"
            )


            try:

                error_data = response.json()

                print(
                    "[ERROR RESPONSE]",
                    error_data,
                )


                # ------------------------------------------
                # IP許可エラー
                # ------------------------------------------

                errors = error_data.get(
                    "errors",
                    {},
                )


                if isinstance(
                    errors,
                    dict,
                ):

                    error_message = errors.get(
                        "errorMessage",
                        "",
                    )

                else:

                    error_message = ""


                if error_message == "CLIENT_IP_NOT_ALLOWED":

                    print(
                        "[ERROR] 楽天APIで"
                        "CLIENT_IP_NOT_ALLOWED"
                        "が返されました"
                    )

                    print(
                        "[ERROR] 現在のIPv4:",
                        get_current_ipv4(),
                    )

                    print(
                        "[ERROR] 再試行せず終了します"
                    )

                    return None


            except ValueError:

                print(
                    "[ERROR RESPONSE]",
                    response.text[:1000],
                )


        except requests.exceptions.ReadTimeout:

            print(
                f"[ERROR] 読み込みタイムアウト "
                f"({retry_count}/{MAX_RETRIES})"
            )


        except requests.exceptions.ConnectTimeout:

            print(
                f"[ERROR] 接続タイムアウト "
                f"({retry_count}/{MAX_RETRIES})"
            )


        except requests.exceptions.ConnectionError as e:

            print(
                f"[ERROR] 接続エラー "
                f"({retry_count}/{MAX_RETRIES}): {e}"
            )


        except requests.RequestException as e:

            print(
                f"[ERROR] リクエストエラー "
                f"({retry_count}/{MAX_RETRIES}): {e}"
            )


        # 最終試行でなければ待機して再試行
        if retry_count < MAX_RETRIES:

            print(
                f"[RETRY] {RETRY_INTERVAL}秒後に"
                "同じページを再試行します"
            )

            time.sleep(
                RETRY_INTERVAL
            )


    print(
        f"[ERROR] Page {page} を"
        f"{MAX_RETRIES}回取得できませんでした"
    )

    return None


# ==================================================
# 画像URL取得
# ==================================================

def get_image_url(
    item: dict,
) -> str:

    image_urls = item.get(
        "mediumImageUrls",
        [],
    )


    if not image_urls:
        return ""


    first_image = image_urls[0]


    if isinstance(
        first_image,
        str,
    ):

        return first_image


    if isinstance(
        first_image,
        dict,
    ):

        return first_image.get(
            "imageUrl",
            "",
        )


    return ""


# ==================================================
# DB保存
# ==================================================

def save_product(
    data: dict,
) -> None:

    conn = sqlite3.connect(
        DB_PATH
    )


    try:

        cur = conn.cursor()


        sql = f"""
        INSERT INTO {table_name}
        (
            shop_name,
            shop_product_id,
            category,
            machine_name,
            product_url,
            image_url,
            price,
            created_at
        )
        VALUES
        (?, ?, ?, ?, ?, ?, ?, ?)
        """


        cur.execute(
            sql,
            (
                SHOP_NAME,
                SHOP_PRODUCT_ID,
                CATEGORY,
                data["machine_name"],
                data["product_url"],
                data["image_url"],
                data["price"],
                datetime.now(),
            ),
        )


        conn.commit()


    except Exception:

        conn.rollback()

        raise


    finally:

        conn.close()


# ==================================================
# API取得開始
# ==================================================

print(
    "[INFO] IPv4固定で実行します"
)

print(
    "[INFO] 現在のIPv4:",
    get_current_ipv4(),
)


session = open_session()

total_count = 0


try:

    # ==================================================
    # ページループ
    # ==================================================

    for page in range(
        1,
        MAX_PAGE + 1,
    ):

        print(
            f"\n{'=' * 80}"
        )

        print(
            f"Page {page}"
        )

        print(
            f"検索キーワード : {KEYWORD}"
        )


        # --------------------------------------------------
        # API取得
        # --------------------------------------------------

        response = get_page(
            session,
            page,
        )


        if response is None:

            print(
                f"[ERROR] Page {page} の取得に"
                "失敗したため終了します"
            )

            break


        # --------------------------------------------------
        # JSON変換
        # --------------------------------------------------

        try:

            data = response.json()

        except ValueError as e:

            print(
                f"[ERROR] JSON解析失敗: {e}"
            )

            print(
                response.text[:2000]
            )

            break


        # --------------------------------------------------
        # 商品一覧
        # --------------------------------------------------

        # 20260701 APIの実レスポンスは "Items"
        # 仕様差を考慮して "items" にも対応
        items = (
            data.get("Items")
            or data.get("items")
            or []
        )


        if not isinstance(
            items,
            list,
        ):

            print(
                "[ERROR] 商品データがlistではありません"
            )

            print(
                "[ERROR] レスポンス:",
                data,
            )

            break


        page_item_count = 0


        print(
            f"API取得件数 : {len(items)}"
        )


        if not items:

            print(
                "商品がありません。終了します。"
            )

            break


        # --------------------------------------------------
        # 商品ループ
        # --------------------------------------------------

        for item in items:

            if not isinstance(
                item,
                dict,
            ):

                print(
                    "[SKIP] 商品データがdictではありません:",
                    item,
                )

                continue


            # --------------------------------------------------
            # 商品名
            # --------------------------------------------------

            machine_name = str(
                item.get(
                    "itemName",
                    "",
                )
                or ""
            ).strip()


            if not machine_name:

                print(
                    "[SKIP] 商品名が空です"
                )

                continue


            # --------------------------------------------------
            # 商品URL
            # --------------------------------------------------

            product_url = str(
                item.get(
                    "itemUrl",
                    "",
                )
                or ""
            ).strip()


            # --------------------------------------------------
            # 画像URL
            # --------------------------------------------------

            image_url = get_image_url(
                item
            )


            # --------------------------------------------------
            # 価格
            # --------------------------------------------------

            price = item.get(
                "itemPrice",
                "",
            )


            if price is None:
                price = ""


            # --------------------------------------------------
            # 楽天固有情報
            # --------------------------------------------------

            shop_name = str(
                item.get(
                    "shopName",
                    "",
                )
                or ""
            ).strip()


            shop_code = str(
                item.get(
                    "shopCode",
                    "",
                )
                or ""
            ).strip()


            item_code = str(
                item.get(
                    "itemCode",
                    "",
                )
                or ""
            ).strip()


            # --------------------------------------------------
            # 表示
            # --------------------------------------------------

            print(
                "-" * 80
            )

            print(
                "サイト    :",
                SHOP_NAME,
            )

            print(
                "店舗名    :",
                shop_name,
            )

            print(
                "店舗コード:",
                shop_code,
            )

            print(
                "商品コード:",
                item_code,
            )

            print(
                "カテゴリ  :",
                CATEGORY,
            )

            print(
                "機種名    :",
                machine_name,
            )

            print(
                "商品URL   :",
                product_url,
            )

            print(
                "画像URL   :",
                image_url,
            )

            print(
                "価格      :",
                price,
            )


            # --------------------------------------------------
            # DB保存
            # --------------------------------------------------

            try:

                save_product({
                    "machine_name": machine_name,
                    "product_url": product_url,
                    "image_url": image_url,
                    "price": price,
                })


                print(
                    "[DB] 保存完了"
                )


            except sqlite3.Error as e:

                print(
                    f"[DB ERROR] 保存失敗: {e}"
                )

                continue


            page_item_count += 1
            total_count += 1


        # --------------------------------------------------
        # ページ結果
        # --------------------------------------------------

        print(
            f"\nページ保存件数 : "
            f"{page_item_count}"
        )


        # --------------------------------------------------
        # 最終ページ判定
        # --------------------------------------------------

        # 1ページ最大件数未満なら最終ページ
        if len(items) < HITS:

            print(
                "[INFO] 取得件数がHITS未満のため"
                "最終ページと判断します"
            )

            break


        # --------------------------------------------------
        # 次ページ待機
        # --------------------------------------------------

        if page < MAX_PAGE:

            print(
                f"[WAIT] 次のページまで"
                f"{REQUEST_INTERVAL}秒待機します"
            )

            time.sleep(
                REQUEST_INTERVAL
            )


finally:

    # ==================================================
    # 終了
    # ==================================================

    session.close()


# ==================================================
# 結果表示
# ==================================================

print(
    "\n" + "=" * 80
)

print(
    f"総取得件数 : {total_count}"
)

print(
    "=" * 80
)


end_time = time.time()


print(
    "[INFO] スクリプト完了"
    f"（実行時間: "
    f"{end_time - start_time:.2f} 秒）"
)